<h2>1. Setup</h2>

In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
RAW_DIR = "../data/raw" 
INTERIM_DIR = "../data/interim" 

os.makedirs(INTERIM_DIR, exist_ok = True)

print("RAW_DIR", os.path.abspath(RAW_DIR))
print("INTERIM_DIR : ",os.path.abspath(INTERIM_DIR))

RAW_DIR C:\Users\USER\Project-3-air-quality-classification\data\raw
INTERIM_DIR :  C:\Users\USER\Project-3-air-quality-classification\data\interim


<h2>2. Helper Functions</h2> 
<h4>2.1 Validasi Kolom</h4>

In [3]:
from typing import List

START_DATE = "2023-01-01"
END_DATE = "2025-10-31"

def assert_columns(df : pd.DataFrame, required : List[str], name : str):
    missing = [c for c in required if c not in df.columns]
    if missing :
        raise ValueError(f"[{name}] Missing required columns : {missing}\nAvailable : {list(df.columns)}")

<h4>2.2 Parsing Tanggal</h4>

In [4]:
def to_datetime_safe(df : pd.DataFrame, col : str, name : str) :
    if col not in df.columns :
        raise ValueError(f"[{name}] date column '{col}' not found.")
    df[col] = pd.to_datetime(df[col], errors = "coerce")
    return df

<h4>2.3 Filtering Rentang Waktu</h4>

In [5]:
def filter_date_range(df: pd.DataFrame, date_col: str = "date"):
    d = df[date_col]
    if isinstance(d, pd.DataFrame):
        d = d.iloc[:, 0]

    d = pd.to_datetime(d, errors="coerce")

    start = pd.to_datetime(START_DATE)
    end = pd.to_datetime(END_DATE)

    mask = d.between(start, end)
    return df.loc[mask].copy()


<h4>2.4 Normalisasi String</h4>

In [6]:
def normalize_text(s : pd.Series) :
    return (s.astype(str)
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", " ", regex = True))

<h4>2.5 Normalisasi String Station</h4>

In [7]:
def drop_duplicates(df : pd.DataFrame, subset : List[str], name : str) :
    before = len(df)
    df = df.drop_duplicates(subset = subset, keep = "first").copy()
    after = len(df)
    print(f"[{name}] drop_duplicates on {subset} : {before} -> {after} (removed {before-after})")
    return df

<h2>3. Load Data - Ground (Label)</h2>

In [8]:
GROUND_PATH = os.path.join(RAW_DIR, "Data_8Station_Ground.csv")

print("GROUND_PATH:", GROUND_PATH)

GROUND_PATH: ../data/raw\Data_8Station_Ground.csv


In [9]:
df_ground_raw = pd.read_csv(GROUND_PATH)
df_ground_raw.head()

,id,data_period,date,station_key,PM10,PM25,SO2,CO,O3,NO2,max,critical_pol,category
0,1,202301,1/1/2023,Bundaran Hotel Indonesia,44,55,47,10,24,9,55,PM25,MEDIUM
1,2,202301,1/2/2023,Bundaran Hotel Indonesia,32,43,52,9,24,8,52,SO2,MEDIUM
2,3,202301,1/3/2023,Bundaran Hotel Indonesia,31,35,49,9,12,7,49,SO2,GOOD
3,4,202301,1/4/2023,Bundaran Hotel Indonesia,30,47,53,11,15,9,53,SO2,MEDIUM
4,5,202301,1/5/2023,Bundaran Hotel Indonesia,38,50,50,13,26,11,50,PM25,GOOD


In [10]:
print(df_ground_raw.shape)
print(df_ground_raw.columns)
df_ground_raw.info()

(8125, 13)
Index(['id', 'data_period', 'date', 'station_key', 'PM10', 'PM25', 'SO2', 'CO',
       'O3', 'NO2', 'max', 'critical_pol', 'category'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8125 entries, 0 to 8124
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            8125 non-null   int64 
 1   data_period   8125 non-null   int64 
 2   date          8125 non-null   object
 3   station_key   8125 non-null   object
 4   PM10          8047 non-null   object
 5   PM25          8112 non-null   object
 6   SO2           8115 non-null   object
 7   CO            8123 non-null   object
 8   O3            8122 non-null   object
 9   NO2           8115 non-null   object
 10  max           8124 non-null   object
 11  critical_pol  7749 non-null   object
 12  category      8125 non-null   object
dtypes: int64(2), object(11)
memory usage: 825.3+ KB


<h4>3.1 Cleaning Data Ground</h4>

In [11]:
df_ground = df_ground_raw.copy()

rename_map = {
    "station_key" : "station",
}
df_ground = df_ground.rename(columns = rename_map)

# Parse date
df_ground = to_datetime_safe(df_ground, "date", "GROUND")

# Normalisasi Station
if "station" in df_ground.columns :
    df_ground["station"] = normalize_text(df_ground["station"])

#rename 1 tempat
df_ground["station"] = df_ground["station"].replace({
    "komp angkasa pura jakarta": "komp angkasa pura jakarta pusat"
})

#ubah unhealty ke unhealthy
if "category" in df_ground.columns :
    df_ground["category"] = (
        df_ground["category"]
        .astype(str)
        .str.strip()
        .str.upper()
        .replace({
            "UNHEALTY" : "UNHEALTHY"
        })
    )
    
df_ground = filter_date_range(df_ground, "date")

# Drop Duplikasi berbasis key utama
key_cols = [c for c in ["date", "station"] if c in df_ground.columns]
df_ground = drop_duplicates(df_ground, subset = key_cols, name = "GROUND")

df_ground.head()

[GROUND] drop_duplicates on ['date', 'station'] : 8125 -> 8125 (removed 0)


,id,data_period,date,station,PM10,PM25,SO2,CO,O3,NO2,max,critical_pol,category
0,1,202301,2023-01-01,bundaran hotel indonesia,44,55,47,10,24,9,55,PM25,MEDIUM
1,2,202301,2023-01-02,bundaran hotel indonesia,32,43,52,9,24,8,52,SO2,MEDIUM
2,3,202301,2023-01-03,bundaran hotel indonesia,31,35,49,9,12,7,49,SO2,GOOD
3,4,202301,2023-01-04,bundaran hotel indonesia,30,47,53,11,15,9,53,SO2,MEDIUM
4,5,202301,2023-01-05,bundaran hotel indonesia,38,50,50,13,26,11,50,PM25,GOOD


In [12]:
print("Check data : ")
df_ground["category"].value_counts()

Check data : 


category
MEDIUM            3323
UNHEALTHY         2461
GOOD              1958
TIDAK ADA DATA     380
VERY_UNHEALTHY       3
Name: count, dtype: int64

<h4>3.2 Save Data Ground</h4>

In [13]:
GROUND_OUT = os.path.join(INTERIM_DIR, "data_ground_clean.csv")
df_ground.to_csv(GROUND_OUT, index = False)
print("Saved : ", GROUND_OUT, "\nRows : ", len(df_ground))


Saved :  ../data/interim\data_ground_clean.csv 
Rows :  8125


<h4>4. Load Data - Sentinel-5P</h4>

In [19]:
S5P_PATH = os.path.join(RAW_DIR, "Data_8Station_Satelit.csv")
print("S5P_PATH : ", S5P_PATH)

S5P_PATH :  ../data/raw\Data_8Station_Satelit.csv


In [20]:
df_s5p_raw = pd.read_csv(S5P_PATH)
df_s5p_raw.head()

,system:index,satellite_sat,date,name,pollutant,station_id,.geo
0,20230101T050749_20230103T111249_0_0,0.000075,1/1/2023,Bundaran Hotel Indonesia,NO2_sat,Bundaran Hotel Indonesia,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,20230101T050749_20230103T111249_2_0,0.000070,1/1/2023,Jalan BDN II,NO2_sat,Jalan BDN II,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,20230101T050749_20230103T111249_3_0,0.000075,1/1/2023,Kebon Jeruk,NO2_sat,Kebon Jeruk,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,20230101T050749_20230103T111249_4_0,0.000070,1/1/2023,Kelapa Gading,NO2_sat,Kelapa Gading,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,20230101T050749_20230103T111249_5_0,0.000070,1/1/2023,Komp Angkasa Pura Jakarta Pusat,NO2_sat,Komp Angkasa Pura Jakarta Pusat,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


In [21]:
print(df_s5p_raw.shape)
print(df_s5p_raw.columns)
df_s5p_raw.info()

(24282, 7)
Index(['system:index', 'satellite_sat', 'date', 'name', 'pollutant',
       'station_id', '.geo'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24282 entries, 0 to 24281
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   system:index   24282 non-null  object 
 1   satellite_sat  24282 non-null  float64
 2   date           24282 non-null  object 
 3   name           24282 non-null  object 
 4   pollutant      24282 non-null  object 
 5   station_id     24282 non-null  object 
 6   .geo           24282 non-null  object 
dtypes: float64(1), object(6)
memory usage: 1.3+ MB


<h4>4.1 Cleaning Sentinel-5P</h4>

In [22]:
df_s5p = df_s5p_raw.copy()

# Rename kolom station
rename_map = {
    "station_id": "station",
}
df_s5p = df_s5p.rename(columns=rename_map)

#Drop kolom yg tidk diperlukan
drop_cols = [c for c in ["system:index", ".geo"] if c in df_s5p.columns]
df_s5p = df_s5p.drop(columns=drop_cols)

# Parse date, harus datetime
df_s5p = to_datetime_safe(df_s5p, "date", "S5P")

# Normalisasi station & pollutant
if "station" in df_s5p.columns:
    df_s5p["station"] = normalize_text(df_s5p["station"])

if "pollutant" in df_s5p.columns:
    df_s5p["pollutant"] = (
        df_s5p["pollutant"].astype(str).str.strip().str.lower()
    )

# Filter date range
df_s5p = filter_date_range(df_s5p, "date")

# Validasi kolom penting
assert_columns(df_s5p, ["date", "station", "pollutant", "satellite_sat"], "S5P")

# Agregasikan klo ad lbh dari 1 baris untuk (date, station, pollutant)
s5p_agg = (
    df_s5p
    .groupby(["date", "station", "pollutant"], as_index=False)["satellite_sat"]
    .mean()
)

print("Before agg:", df_s5p.shape)
print("After  agg:", s5p_agg.shape)

# Pivot ke wide format
df_s5p_wide = (
    s5p_agg
    .pivot(index=["date", "station"], columns="pollutant", values="satellite_sat")
    .reset_index()
)

# Rapikan nama kolom: tambah prefix s5p_
df_s5p_wide.columns = [
    f"{c}" if c not in ["date", "station"] else c
    for c in df_s5p_wide.columns
]

# Drop duplicates
df_s5p_wide = drop_duplicates(df_s5p_wide, subset=["date", "station"], name="S5P_WIDE")

df_s5p_wide = df_s5p_wide.rename(columns={
        "co_sat" : "s5p_co", 
        "no2_sat" : "s5p_no2", 
        "o3_sat" : "s5p_o3",
        "so2_sat" : "s5p_so2"
    })

print("S5P wide shape:", df_s5p_wide.shape)
df_s5p_wide.head()


Before agg: (24282, 5)
After  agg: (24282, 4)
[S5P_WIDE] drop_duplicates on ['date', 'station'] : 7990 -> 7990 (removed 0)
S5P wide shape: (7990, 6)


,date,station,s5p_co,s5p_no2,s5p_o3,s5p_so2
0,2023-01-01,bundaran hotel indonesia,NaN,0.000075,0.120394,NaN
1,2023-01-01,jagakarsa,NaN,NaN,0.120566,NaN
2,2023-01-01,jalan bdn ii,NaN,0.000070,0.118817,NaN
3,2023-01-01,kebon jeruk,NaN,0.000075,0.120394,NaN
4,2023-01-01,kelapa gading,NaN,0.000070,0.120410,NaN


In [23]:
print("Jumlah station : ", df_s5p_wide["station"].nunique())
print(sorted(df_s5p_wide["station"].unique()))
print("Columns : \n", df_s5p_wide.columns.tolist())


Jumlah station :  8
['bundaran hotel indonesia', 'jagakarsa', 'jalan bdn ii', 'kebon jeruk', 'kelapa gading', 'komp angkasa pura jakarta pusat', 'lubang buaya', 'us consulate jakarta pusat']
Columns : 
 ['date', 'station', 's5p_co', 's5p_no2', 's5p_o3', 's5p_so2']


<h4>4.2 Save Data S5P</h4>

In [24]:
S5P_OUT_PATH = os.path.join(INTERIM_DIR, "data_s5p_clean.csv")
df_s5p_wide.to_csv(S5P_OUT_PATH, index=False)

print("Saved : ", S5P_OUT_PATH, "\nRows : ", len(df_s5p_wide))


Saved :  ../data/interim\data_s5p_clean.csv 
Rows :  7990


<h2>5. Load Data - MODIS</h2>

In [25]:
MODIS_PATH = os.path.join(RAW_DIR, "Jakarta_MODIS_AOD_SPku_2023_2025_fix.csv")
print("MODIS_PATH : ", MODIS_PATH)

MODIS_PATH :  ../data/raw\Jakarta_MODIS_AOD_SPku_2023_2025_fix.csv


In [26]:
df_modis_raw = pd.read_csv(MODIS_PATH)
df_modis_raw.head()

,system:index,MODIS_AOD_047,date,station_id,.geo
0,4_4_0,214.0,2023-01-05,Kelapa_Gading,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,4_5_0,131.0,2023-01-05,Komp_Angkasa_Pura_Jakarta_Pusat,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,5_0_0,115.0,2023-01-06,Bundaran_Hotel_Indonesia,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,5_2_0,152.0,2023-01-06,Jalan_BDN_II,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,5_3_0,209.0,2023-01-06,Kebon_Jeruk,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


<h4>5.1 Cleaning MODIS</h4>

In [27]:
df_modis = df_modis_raw.copy()

df_modis = df_modis.rename(columns={"station_id" : "station"})
df_modis = to_datetime_safe(df_modis, "date", "MODIS")

if "station" in df_modis.columns :
    df_modis["station"] = normalize_text(df_modis["station"]
        .astype(str)                                    
        .str.strip()
        .str.replace("_", " ", regex = False)
        .str.lower()
        .str.replace(r"\s+", " ", regex = True)
    )

df_modis = filter_date_range(df_modis, "date")

key_cols = [c for c in ["date", "station"] if c in df_modis.columns]
df_modis = drop_duplicates(df_modis, subset=key_cols, name = "MODIS")

df_modis.head()

[MODIS] drop_duplicates on ['date', 'station'] : 1782 -> 1782 (removed 0)


,system:index,MODIS_AOD_047,date,station,.geo
0,4_4_0,214.0,2023-01-05,kelapa gading,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,4_5_0,131.0,2023-01-05,komp angkasa pura jakarta pusat,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,5_0_0,115.0,2023-01-06,bundaran hotel indonesia,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,5_2_0,152.0,2023-01-06,jalan bdn ii,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,5_3_0,209.0,2023-01-06,kebon jeruk,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


<h4>5.2 Save Data MODIS</h4>

In [28]:
MODIS_OUT = os.path.join(INTERIM_DIR, "data_modis_clean.csv")
df_modis.to_csv(MODIS_OUT, index = False)
print("Saved : ", MODIS_OUT, "\nrows : ", len(df_modis))

Saved :  ../data/interim\data_modis_clean.csv 
rows :  1782


<h2>6. Load Data - VIIRS</h2>

In [29]:
VIIRS_PATH = os.path.join(RAW_DIR, "Jakarta_VIIRS_NTL_SPku_2023_2025.csv")
print("VIIRS_PATH : ", VIIRS_PATH)

VIIRS_PATH :  ../data/raw\Jakarta_VIIRS_NTL_SPku_2023_2025.csv


In [30]:
df_viirs_raw = pd.read_csv(VIIRS_PATH)
df_viirs_raw.head()

,system:index,VIIRS_NTL,date,station_id,.geo
0,0_0_0,74.275642,2023-01-01,Bundaran_Hotel_Indonesia,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,0_1_0,48.117756,2023-01-01,Jagakarsa,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,0_2_0,36.579357,2023-01-01,Jalan_BDN_II,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,0_3_0,36.949665,2023-01-01,Kebon_Jeruk,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,0_4_0,50.655720,2023-01-01,Kelapa_Gading,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


<h4>6.1 Cleaning VIIRS</h4>

In [31]:
df_viirs = df_viirs_raw.copy()

df_viirs = df_viirs.rename(columns={"station_id": "station"})
df_viirs = to_datetime_safe(df_viirs, "date", "VIIRS")

if "station" in df_viirs.columns:
    df_viirs["station"] = normalize_text(df_viirs["station"]
        .astype(str)                                    
        .str.strip()
        .str.replace("_", " ", regex = False)
        .str.lower()
        .str.replace(r"\s+", " ", regex = True)
    )

df_viirs = filter_date_range(df_viirs, "date")

key_cols = [c for c in ["date", "station"] if c in df_viirs.columns]
df_viirs = drop_duplicates(df_viirs, subset=key_cols, name="VIIRS")

df_viirs.head()


[VIIRS] drop_duplicates on ['date', 'station'] : 8080 -> 8080 (removed 0)


,system:index,VIIRS_NTL,date,station,.geo
0,0_0_0,74.275642,2023-01-01,bundaran hotel indonesia,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,0_1_0,48.117756,2023-01-01,jagakarsa,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,0_2_0,36.579357,2023-01-01,jalan bdn ii,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,0_3_0,36.949665,2023-01-01,kebon jeruk,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,0_4_0,50.655720,2023-01-01,kelapa gading,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


<h4>6.2 Save Data VIIRS</h4>

In [32]:
VIIRS_OUT = os.path.join(INTERIM_DIR, "data_viirs_clean.csv")
df_viirs.to_csv(VIIRS_OUT, index=False)
print("Saved : ", VIIRS_OUT, "\nrows : ", len(df_viirs))


Saved :  ../data/interim\data_viirs_clean.csv 
rows :  8080


<h2>7. Validasi Output</h2>

In [33]:
def quick_validate(df: pd.DataFrame, name: str):
    print("="*60)
    print(name)
    print("Shape:", df.shape)
    print("Date min/max:", df["date"].min(), df["date"].max() if "date" in df.columns else None)
    if "date" in df.columns:
        print("Date min/max:", df["date"].min(), df["date"].max())
    if "station" in df.columns:
        print("Unique stations:", df["station"].nunique())
    print("Missing % (top 10):")
    display((df.isna().mean().sort_values(ascending=False) * 100).head(10))


In [34]:
# Reload output untuk memastikan file benar-benar terbentuk
g = pd.read_csv(os.path.join(INTERIM_DIR, "data_ground_clean.csv"), parse_dates=["date"])
s = pd.read_csv(os.path.join(INTERIM_DIR, "data_s5p_clean.csv"), parse_dates=["date"])
m = pd.read_csv(os.path.join(INTERIM_DIR, "data_modis_clean.csv"), parse_dates=["date"])
v = pd.read_csv(os.path.join(INTERIM_DIR, "data_viirs_clean.csv"), parse_dates=["date"])

quick_validate(g, "GROUND_CLEAN")
quick_validate(s, "S5P_CLEAN")
quick_validate(m, "MODIS_CLEAN")
quick_validate(v, "VIIRS_CLEAN")


GROUND_CLEAN
Shape: (8125, 13)
Date min/max: 2023-01-01 00:00:00 2025-10-31 00:00:00
Date min/max: 2023-01-01 00:00:00 2025-10-31 00:00:00
Unique stations: 8
Missing % (top 10):


critical_pol    4.627692
PM10            0.960000
PM25            0.160000
SO2             0.123077
NO2             0.123077
O3              0.036923
CO              0.024615
max             0.012308
id              0.000000
station         0.000000
dtype: float64

S5P_CLEAN
Shape: (7990, 6)
Date min/max: 2023-01-01 00:00:00 2025-10-30 00:00:00
Date min/max: 2023-01-01 00:00:00 2025-10-30 00:00:00
Unique stations: 8
Missing % (top 10):


s5p_so2    50.025031
s5p_co     35.682103
s5p_no2     9.299124
s5p_o3      1.088861
station     0.000000
date        0.000000
dtype: float64

MODIS_CLEAN
Shape: (1782, 5)
Date min/max: 2023-01-05 00:00:00 2025-10-26 00:00:00
Date min/max: 2023-01-05 00:00:00 2025-10-26 00:00:00
Unique stations: 8
Missing % (top 10):


system:index     0.0
MODIS_AOD_047    0.0
date             0.0
station          0.0
.geo             0.0
dtype: float64

VIIRS_CLEAN
Shape: (8080, 5)
Date min/max: 2023-01-01 00:00:00 2025-10-30 00:00:00
Date min/max: 2023-01-01 00:00:00 2025-10-30 00:00:00
Unique stations: 8
Missing % (top 10):


VIIRS_NTL       1.881188
system:index    0.000000
date            0.000000
station         0.000000
.geo            0.000000
dtype: float64

<h2>8. Merge Dataset</h2>

<h4>8.1 Load Data</h4>

In [35]:
# ====== 5. MERGE CLEANED DATASETS ======

GROUND_CLEAN_PATH = os.path.join(INTERIM_DIR, "data_ground_clean.csv")
S5P_CLEAN_PATH    = os.path.join(INTERIM_DIR, "data_s5p_clean.csv")
MODIS_CLEAN_PATH  = os.path.join(INTERIM_DIR, "data_modis_clean.csv")
VIIRS_CLEAN_PATH  = os.path.join(INTERIM_DIR, "data_viirs_clean.csv")

for p in [GROUND_CLEAN_PATH, S5P_CLEAN_PATH, MODIS_CLEAN_PATH, VIIRS_CLEAN_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"File clean tidak ditemukan: {p}")

ground = pd.read_csv(GROUND_CLEAN_PATH)
s5p    = pd.read_csv(S5P_CLEAN_PATH)
modis  = pd.read_csv(MODIS_CLEAN_PATH)
viirs  = pd.read_csv(VIIRS_CLEAN_PATH)

print("ground:", ground.shape)
print("s5p   :", s5p.shape)
print("modis :", modis.shape)
print("viirs :", viirs.shape)

ground: (8125, 13)
s5p   : (7990, 6)
modis : (1782, 5)
viirs : (8080, 5)


<h4>8.2 Validasi Minimal Kolom Kunci</h4>

In [36]:
KEY_COLS = ["date", "station"]

assert_columns(ground, KEY_COLS, "GROUND_CLEAN")
assert_columns(s5p,    KEY_COLS, "S5P_CLEAN")
assert_columns(modis,  KEY_COLS, "MODIS_CLEAN")
assert_columns(viirs,  KEY_COLS, "VIIRS_CLEAN")

print("OK: kolom kunci tersedia di semua dataset:", KEY_COLS)


OK: kolom kunci tersedia di semua dataset: ['date', 'station']


<h4>8.3 Audit Duplikasi Key</h4>

In [37]:
def check_key_duplicates(df, name):
    dup = df.duplicated(subset=KEY_COLS).sum()
    print(f"[{name}] duplicates by {KEY_COLS}: {dup}")
    if dup > 0:
        display(df[df.duplicated(subset=KEY_COLS, keep=False)]
                .sort_values(KEY_COLS)
                .head(20))

check_key_duplicates(ground, "GROUND_CLEAN")
check_key_duplicates(s5p,    "S5P_CLEAN")
check_key_duplicates(modis,  "MODIS_CLEAN")
check_key_duplicates(viirs,  "VIIRS_CLEAN")


[GROUND_CLEAN] duplicates by ['date', 'station']: 0
[S5P_CLEAN] duplicates by ['date', 'station']: 0
[MODIS_CLEAN] duplicates by ['date', 'station']: 0
[VIIRS_CLEAN] duplicates by ['date', 'station']: 0


<h4>8.4 Tambah Prefix Fitur Satelit</h4>

In [38]:
def add_prefix_except_key(df, prefix, key_cols=KEY_COLS):
    df = df.copy()
    rename_map = {c: f"{prefix}{c}" for c in df.columns if c not in key_cols}
    return df.rename(columns=rename_map)

s5p_p   = add_prefix_except_key(s5p,   "")
modis_p = add_prefix_except_key(modis, "modis_")
viirs_p = add_prefix_except_key(viirs, "viirs_")

print("Contoh kolom setelah prefix:")
print("S5P  :", [c for c in s5p_p.columns if c.startswith("")][:10])
print("MODIS:", [c for c in modis_p.columns if c.startswith("modis_")][:10])
print("VIIRS:", [c for c in viirs_p.columns if c.startswith("viirs_")][:10])


Contoh kolom setelah prefix:
S5P  : ['date', 'station', 's5p_co', 's5p_no2', 's5p_o3', 's5p_so2']
MODIS: ['modis_system:index', 'modis_MODIS_AOD_047', 'modis_.geo']
VIIRS: ['viirs_system:index', 'viirs_VIIRS_NTL', 'viirs_.geo']


<h4>8.5 Merge Data</h4>

In [39]:
merged = ground.merge(s5p_p,   on=KEY_COLS, how="left")
merged = merged.merge(modis_p, on=KEY_COLS, how="left")
merged = merged.merge(viirs_p, on=KEY_COLS, how="left")

print("Merged shape:", merged.shape)
merged.head()

Merged shape: (8125, 23)


,id,data_period,date,station,PM10,PM25,SO2,CO,O3,NO2,...,s5p_co,s5p_no2,s5p_o3,s5p_so2,modis_system:index,modis_MODIS_AOD_047,modis_.geo,viirs_system:index,viirs_VIIRS_NTL,viirs_.geo
0,1,202301,2023-01-01,bundaran hotel indonesia,44,55,47,10,24,9,...,NaN,0.000075,0.120394,NaN,NaN,NaN,NaN,0_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
1,2,202301,2023-01-02,bundaran hotel indonesia,32,43,52,9,24,8,...,NaN,0.000097,0.119363,NaN,NaN,NaN,NaN,1_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
2,3,202301,2023-01-03,bundaran hotel indonesia,31,35,49,9,12,7,...,NaN,NaN,0.120255,NaN,NaN,NaN,NaN,2_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
3,4,202301,2023-01-04,bundaran hotel indonesia,30,47,53,11,15,9,...,NaN,0.000140,0.118994,NaN,NaN,NaN,NaN,3_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."
4,5,202301,2023-01-05,bundaran hotel indonesia,38,50,50,13,26,11,...,0.032939,0.000123,0.119520,0.00017,NaN,NaN,NaN,4_0_0,74.275642,"{""geodesic"":false,""type"":""Point"",""coordinates""..."


<h4>5.6 Audit Hasil Merge</h4>

In [40]:
def match_rate(base_df, merged_df, prefix):
    cols = [c for c in merged_df.columns if c.startswith(prefix)]
    if len(cols) == 0:
        return None
    # pakai minimal 1 kolom satelit sebagai indikator match
    indicator_col = cols[0]
    rate = merged_df[indicator_col].notna().mean() * 100
    return indicator_col, rate

for pref in ["s5p_", "modis_", "viirs_"]:
    res = match_rate(ground, merged, pref)
    if res is None:
        print(f"{pref}: tidak ada kolom (cek prefix).")
    else:
        ind_col, rate = res
        print(f"{pref}: match rate ~ {rate:.2f}% (indikator: {ind_col})")


s5p_: match rate ~ 62.26% (indikator: s5p_co)
modis_: match rate ~ 21.48% (indikator: modis_system:index)
viirs_: match rate ~ 97.85% (indikator: viirs_system:index)


In [41]:
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

MERGED_OUT_PATH = os.path.join(PROCESSED_DIR, "air_quality_merged_clean.csv")
merged.to_csv(MERGED_OUT_PATH, index=False)

print("Saved merged dataset:", MERGED_OUT_PATH)
print("Final columns:", len(merged.columns))


Saved merged dataset: ../data/processed\air_quality_merged_clean.csv
Final columns: 23
